# Código para extraer mensajes de meta

# Descarga completa comentarios de facebook

In [ ]:
import requests
import pandas as pd
import time

# --- CONFIGURACIÓN ---
ACCESS_TOKEN = "EAAQ9FrwwOUABRLiqbGSRxot7S5nTdvyJFaFJOkgdivS3bNlYzfZCvItDpJDrZC9ZA6tAH50gfFJgEVKIx3Wh3zuc6FWA178PA7rq836Kbo3kAaNn8YZBj9W0huVrPQaqGx8yum5NemEAWRf8lre2eIbnJpEaSS909VNjYvBrbII7wGPGhxR54tycbZCmjZAXOFl5GHscTPhYYbvgCwUPu3QnraRS31uvNzobbZCPzh47aWj40UeAJWpIwVGjWCln7YWB0JdQIujlMov6N0JuxZA4eMNdXQZDZD"

# ACCESS_TOKEN = 'EAAQ9FrwwOUABQdWinGZCvfDqvknse4Kk8TJRWbH94C7AlZAuARFB0toGPh7tKeK17BNhtRJK23CULZC6ioBIYG4HE1LeacXiXMMhdAYYx2nIpTLGQBx2ZBpoHl1HdIbwiBhxwS65qimNubLRgYQJAZCVZABg45kLaZBPdIIZB6ikDC7tEgn2kr0KLZAkguZCAssnZAlhhfT5ZBKe81ZARadxGLqDNP0u4wfU5EioxBPCrl0UZD'
PAGE_ID = '895431323874532'

def obtener_todos_los_videos(token):
    print("Buscando lista de videos...")
    videos = []
    # Endpoint para videos subidos
    url = f"https://graph.facebook.com/v24.0/{PAGE_ID}/videos?fields=id,description,created_time&limit=50&access_token={token}"
    
    while url:
        res = requests.get(url).json()
        if 'data' in res:
            videos.extend(res['data'])
        url = res.get('paging', {}).get('next')
        print(f"Videos encontrados hasta ahora: {len(videos)}")
    return videos

def obtener_comentarios_de_video(video_id, token):
    comentarios = []
    url = f"https://graph.facebook.com/v24.0/{video_id}/comments?fields=from,message,created_time&limit=100&access_token={token}"
    
    while url:
        res = requests.get(url).json()
        if 'data' in res:
            for c in res['data']:
                comentarios.append({
                    'Video_ID': video_id,
                    'Fecha_Comentario': c.get('created_time'),
                    'Usuario': c.get('from', {}).get('name', 'Anónimo'),
                    'Mensaje': c.get('message')
                })
        url = res.get('paging', {}).get('next')
    return comentarios

# --- EJECUCIÓN PRINCIPAL ---
todos_los_videos = obtener_todos_los_videos(ACCESS_TOKEN)
base_datos_comentarios = []

print(f"\nIniciando descarga de comentarios de {len(todos_los_videos)} videos...")

for i, video in enumerate(todos_los_videos):
    v_id = video['id']
    desc = video.get('description', 'Sin descripción')[:50] # Tomamos los primeros 50 caracteres
    
    print(f"[{i+1}/{len(todos_los_videos)}] Procesando video: {v_id} ({desc}...)")
    
    comentarios_video = obtener_comentarios_de_video(v_id, ACCESS_TOKEN)
    base_datos_comentarios.extend(comentarios_video)
    
    # Pequeña pausa para no saturar la API (Rate Limit)
    time.sleep(0.5)

# Guardar todo en un Excel
if base_datos_comentarios:
    df = pd.DataFrame(base_datos_comentarios)
    df.to_excel('HISTORICO_COMENTARIOS_VIDEOS.xlsx', index=False)
    print(f"\n✅ ¡TERMINADO! Se guardaron {len(base_datos_comentarios)} comentarios en 'HISTORICO_COMENTARIOS_VIDEOS.xlsx'")
else:
    print("\n❌ No se encontraron comentarios.")

In [13]:
import requests
import pandas as pd
import time
from datetime import datetime
# --- CONFIGURACIÓN ---
ACCESS_TOKEN = "EAAQ9FrwwOUABRLhMBRceVdgfV0CPALrTGSJCMh5zyc6LPXLBBq6Gch7ZCdQrZBun9mBMwjXCE8AZA2J6PfgzhKJjWPMTDuaq1DZCuwmlvV12S3zPvvRWEql9EeHrGNDs1wcm6ehMigoG2aGRIiZBA8VXilvOzZAHxdkJDCKvAYAMCByYWT3WgBms8kpMcXLPk6Td7NUYtZCp6Ho55Wr7ZAoaTVlL6Dd6euzOVZAY8cvcixslH"

# ACCESS_TOKEN = 'EAAQ9FrwwOUABQdWinGZCvfDqvknse4Kk8TJRWbH94C7AlZAuARFB0toGPh7tKeK17BNhtRJK23CULZC6ioBIYG4HE1LeacXiXMMhdAYYx2nIpTLGQBx2ZBpoHl1HdIbwiBhxwS65qimNubLRgYQJAZCVZABg45kLaZBPdIIZB6ikDC7tEgn2kr0KLZAkguZCAssnZAlhhfT5ZBKe81ZARadxGLqDNP0u4wfU5EioxBPCrl0UZD'
PAGE_ID = '895431323874532'
# Definir el inicio del año actual en formato Unix Timestamp
# Esto le dice a Facebook: "Solo dame datos desde el 1 de enero"
inicio_anio = datetime(datetime.now().year, 1, 1)
since_timestamp = int(inicio_anio.timestamp())

def obtener_todos_los_videos(token):
    print(f"Buscando videos desde el {inicio_anio.date()}...")
    videos = []
    # Añadimos &since={since_timestamp} para filtrar videos subidos este año
    url = f"https://graph.facebook.com/v24.0/{PAGE_ID}/videos?fields=id,description,created_time&limit=50&since={since_timestamp}&access_token={token}"
    
    while url:
        res = requests.get(url).json()
        if 'data' in res:
            videos.extend(res['data'])
        url = res.get('paging', {}).get('next')
        print(f"Videos encontrados hasta ahora: {len(videos)}")
    return videos

def obtener_comentarios_de_video(video_id, token):
    comentarios = []
    # Eliminamos 'from' temporalmente para probar si es un problema de permisos
    url = f"https://graph.facebook.com/v24.0/{video_id}/comments?fields=message,created_time&limit=100&since={since_timestamp}&access_token={token}"
    
    try:
        res = requests.get(url).json()
        
        # DEBUG: Imprimir si la API devuelve un error
        if 'error' in res:
            print(f"   ⚠️ Error en video {video_id}: {res['error'].get('message')}")
            return []

        if 'data' in res:
            for c in res['data']:
                comentarios.append({
                    'Video_ID': video_id,
                    'Fecha_Comentario': c.get('created_time'),
                    'Usuario': 'Verificar Permisos', # Marcador de posición
                    'Mensaje': c.get('message')
                })
            
            # Si el video tiene comentarios, avisar en consola
            if len(res['data']) > 0:
                print(f"   ✅ Se encontraron {len(res['data'])} comentarios.")
                
    except Exception as e:
        print(f"   ❌ Error de conexión: {e}")
        
    return comentarios

# --- EJECUCIÓN PRINCIPAL ---
todos_los_videos = obtener_todos_los_videos(ACCESS_TOKEN)
base_datos_comentarios = []

if not todos_los_videos:
    print("No se encontraron videos nuevos este año. Buscando comentarios en videos existentes...")
    # Opcional: Si quieres buscar comentarios nuevos en videos antiguos, 
    # tendrías que cargar una lista previa de IDs de videos.

print(f"\nIniciando descarga de comentarios de {len(todos_los_videos)} videos...")

for i, video in enumerate(todos_los_videos):
    v_id = video['id']
    desc = video.get('description', 'Sin descripción')[:50]
    
    print(f"[{i+1}/{len(todos_los_videos)}] Procesando video: {v_id} ({desc}...)")
    
    comentarios_video = obtener_comentarios_de_video(v_id, ACCESS_TOKEN)
    base_datos_comentarios.extend(comentarios_video)
    
    time.sleep(0.5)

# Guardar resultados
if base_datos_comentarios:
    df = pd.DataFrame(base_datos_comentarios)
    # Formatear la fecha para que sea más legible en Excel
    df['Fecha_Comentario'] = pd.to_datetime(df['Fecha_Comentario']).dt.tz_localize(None)
    
    nombre_archivo = f'COMENTARIOS_{datetime.now().year}.xlsx'
    df.to_excel(nombre_archivo, index=False)
    print(f"\n✅ ¡TERMINADO! Se guardaron {len(base_datos_comentarios)} comentarios en '{nombre_archivo}'")
else:
    print("\n❌ No se encontraron comentarios nuevos en el periodo seleccionado.")

Buscando videos desde el 2026-01-01...
Videos encontrados hasta ahora: 50
Videos encontrados hasta ahora: 100
Videos encontrados hasta ahora: 150
Videos encontrados hasta ahora: 166

Iniciando descarga de comentarios de 166 videos...
[1/166] Procesando video: 1478203890672124 (MAÑANA SÚPER PROMO POCIÓN 🤑 🧮

Haz los cálculos y ...)
   ✅ Se encontraron 3 comentarios.
[2/166] Procesando video: 2666626913706852 (Sin descripción...)
[3/166] Procesando video: 872582972456007 (Sin descripción...)
[4/166] Procesando video: 979789037831544 (Sin descripción...)
[5/166] Procesando video: 797981066701756 (Sin descripción...)
[6/166] Procesando video: 1511733030465891 (Sin descripción...)
[7/166] Procesando video: 1937211926937008 (Sin descripción...)
[8/166] Procesando video: 4144910759105361 (Sin descripción...)
[9/166] Procesando video: 949092094616199 (Sin descripción...)
[10/166] Procesando video: 2846930322313394 (Sin descripción...)
[11/166] Procesando video: 1683096206442328 (Sin descripció

# Instagran Completo

In [14]:
import requests
import pandas as pd
import time

# --- TUS DATOS ---
ACCESS_TOKEN = 'EAAQ9FrwwOUABQiVgUJzxNVfpEfoW22ukZBGn21izTqT7oZBFUcoJu0k584WDIQPS2XuAqZAZByWyE2rb5kpq5ZBc2JreZCNNHV1Q1cC4pFlZBRQ2Ov6nySEiOXtrVQZAIhkAcjVAPh4DoZBmVDnSectPM42AC157AV82Ou9vS2t9hk7U0MWeYrpRrTxcW20QeksjR2Cqb7uImw4jh2CZC9k3oPTPsUNGxlxy9nRm9D2cmsGd9PtupaZBjIwtbEDRxskbvaImjfNzakaMRJ7VPDK8eZCQR5j0'
# Asegúrate de usar el IG User ID (no el de Facebook)
IG_USER_ID = '17841402208128553' 

def descargar_instagram_completo():
    # 1. Solicitamos 50 posts por lote para ir más rápido (limit=50)
    url_media = f"https://graph.facebook.com/v24.0/{IG_USER_ID}/media?fields=id,caption,media_product_type,timestamp&limit=50&access_token={ACCESS_TOKEN}"
    
    todos_los_datos = []
    contador_posts = 0
    
    print("🚀 INICIANDO DESCARGA DEL HISTORIAL COMPLETO...")

    # --- BUCLE EXTERNO: Recorre las páginas de PUBLICACIONES ---
    while url_media:
        try:
            response = requests.get(url_media)
            data_media = response.json()
            
            if 'error' in data_media:
                print(f"❌ Error en API: {data_media['error']['message']}")
                break

            posts_batch = data_media.get('data', [])
            
            if not posts_batch:
                print("No hay más publicaciones.")
                break

            # Procesamos cada post del lote actual
            for post in posts_batch:
                contador_posts += 1
                media_id = post['id']
                caption = post.get('caption', 'Sin descripción')[:40].replace('\n', ' ')
                tipo = post.get('media_product_type')
                
                print(f"[{contador_posts}] Analizando: {caption}...")
                
                # --- BUCLE INTERNO: Recorre los COMENTARIOS de ese post ---
                url_comments = f"https://graph.facebook.com/v24.0/{media_id}/comments?fields=text,username,timestamp&limit=100&access_token={ACCESS_TOKEN}"
                
                while url_comments:
                    res_comm = requests.get(url_comments).json()
                    comments_data = res_comm.get('data', [])
                    
                    for c in comments_data:
                        todos_los_datos.append({
                            'ID_Post': media_id,
                            'Fecha_Post': post.get('timestamp'),
                            'Tipo_Post': tipo,
                            'Descripción_Post': post.get('caption'),
                            'Usuario_Comentario': c.get('username'),
                            'Comentario': c.get('text'),
                            'Fecha_Comentario': c.get('timestamp')
                        })
                    
                    # Paginación de COMENTARIOS (Siguiente página de comentarios)
                    url_comments = res_comm.get('paging', {}).get('next')

            # --- Paginación de PUBLICACIONES (Siguiente lote de 50 posts) ---
            url_media = data_media.get('paging', {}).get('next')
            
            if url_media:
                print("⏳ Cargando posts más antiguos... (Espera unos segundos)")
                time.sleep(1) # Pausa pequeña para cuidar el límite de la API
                
        except Exception as e:
            print(f"⚠️ Ocurrió un error inesperado: {e}")
            break

    return todos_los_datos

# --- EJECUCIÓN ---
datos_finales = descargar_instagram_completo()

if datos_finales:
    print(f"\n✅ PROCESO FINALIZADO.")
    print(f"📊 Se extrajeron {len(datos_finales)} comentarios en total.")
    
    # Guardar en Excel
    df = pd.DataFrame(datos_finales)
    df.to_excel('Instagram_Historial_Completo.xlsx', index=False)
    print("💾 Archivo guardado como: Instagram_Historial_Completo.xlsx")
else:
    print("❌ No se encontraron datos o hubo un error.")

🚀 INICIANDO DESCARGA DEL HISTORIAL COMPLETO...
❌ Error en API: Error validating access token: Session has expired on Tuesday, 20-Jan-26 14:00:00 PST. The current time is Wednesday, 15-Apr-26 06:29:39 PDT.
❌ No se encontraron datos o hubo un error.


# Instagran por año


In [15]:
import requests
import pandas as pd
import time
from datetime import datetime

# --- TUS DATOS ---
ACCESS_TOKEN = 'EAAQ9FrwwOUABRNCpUim5u3kvCZC9R4v2J9NXjIhbnRkCeVSYJ560bxQFYd1VrEvLYhthSnlB6DKmEqNQsPQBvHbiftPVFYcEQUjCZAEYSHbkfRqpq3ANBskel9GdYRN43aW9ltBmlziaWPFOObfa5BhjpIL60CXfUexsoSJkpL5lrOJfMkxdKphareMkpOSwXsf2q0Rar7MSi5aAJMCldvWGyKH146ihNcnEH9ZAesGMYizQCL9bmj71596jl9sxN3SJ5Vq8CGFlsbW'
IG_USER_ID = '17841402208128553' 
ANIO_OBJETIVO = 2026 # Cambia a 2025 si quieres el anterior

def descargar_instagram_2026():
    # Solicitamos posts con el campo timestamp
    url_media = f"https://graph.facebook.com/v24.0/{IG_USER_ID}/media?fields=id,caption,media_product_type,timestamp&limit=50&access_token={ACCESS_TOKEN}"
    
    todos_los_datos = []
    print(f"🚀 INICIANDO EXTRACCIÓN DEL AÑO {ANIO_OBJETIVO}...")

    while url_media:
        try:
            response = requests.get(url_media)
            data_media = response.json()
            
            if 'error' in data_media:
                print(f"❌ Error: {data_media['error']['message']}")
                break

            posts_batch = data_media.get('data', [])
            if not posts_batch: break

            for post in posts_batch:
                # Convertir fecha del post (ej: 2026-03-15T...) a objeto datetime
                fecha_post = datetime.strptime(post['timestamp'], "%Y-%m-%dT%H:%M:%S%z")
                
                # SI EL POST ES DE UN AÑO ANTERIOR AL OBJETIVO, PARAMOS EL BUCLE
                if fecha_post.year < ANIO_OBJETIVO:
                    print(f"📍 Se alcanzó el año {fecha_post.year}. Deteniendo búsqueda.")
                    url_media = None # Rompe el bucle exterior
                    break
                
                # SI EL POST ES DEL AÑO OBJETIVO, BUSCAMOS COMENTARIOS
                if fecha_post.year == ANIO_OBJETIVO:
                    media_id = post['id']
                    print(f"🔎 Analizando post de {fecha_post.strftime('%d/%m/%Y')}: {post.get('caption', '')[:30]}...")
                    
                    url_comments = f"https://graph.facebook.com/v24.0/{media_id}/comments?fields=text,username,timestamp&limit=100&access_token={ACCESS_TOKEN}"
                    
                    while url_comments:
                        res_comm = requests.get(url_comments).json()
                        for c in res_comm.get('data', []):
                            todos_los_datos.append({
                                'ID_Post': media_id,
                                'Fecha_Post': post.get('timestamp'),
                                'Usuario': c.get('username'),
                                'Comentario': c.get('text'),
                                'Fecha_Comentario': c.get('timestamp')
                            })
                        url_comments = res_comm.get('paging', {}).get('next')

            if url_media:
                url_media = data_media.get('paging', {}).get('next')
                time.sleep(0.5)
                
        except Exception as e:
            print(f"⚠️ Error: {e}")
            break

    return todos_los_datos

# Ejecución y guardado
datos = descargar_instagram_2026()
if datos:
    df = pd.DataFrame(datos)
    df.to_excel(f'Comentarios_IG_{ANIO_OBJETIVO}.xlsx', index=False)
    print(f"✅ ¡Listo! {len(datos)} comentarios guardados.")

🚀 INICIANDO EXTRACCIÓN DEL AÑO 2026...
🔎 Analizando post de 14/04/2026: MAÑANA SÚPER PROMO POCIÓN 🤑 🧮
...
🔎 Analizando post de 12/04/2026: El acceso será desbloqueado… n...
🔎 Analizando post de 11/04/2026: ¿Qué hace increíble a @greeicy...
🔎 Analizando post de 09/04/2026: Hoy se acabaron para siempre l...
🔎 Analizando post de 08/04/2026: Esto NO es real 🫣 pero que cre...
🔎 Analizando post de 07/04/2026: Cómo ganar con el efecto Greei...
🔎 Analizando post de 06/04/2026: Tu pelo no es de una sola nece...
🔎 Analizando post de 05/04/2026: Hay cosas que son difíciles de...
🔎 Analizando post de 04/04/2026: Nuestras famosas están enamora...
🔎 Analizando post de 01/04/2026: Muchachas, el camino no ha sid...
🔎 Analizando post de 30/03/2026: 5 Shampoo pero… ¿qué hace cada...
🔎 Analizando post de 28/03/2026: No lo sabíamos pero lo sospech...
🔎 Analizando post de 27/03/2026: En estas vacaciones, sol + clo...
🔎 Analizando post de 26/03/2026: Esta Mascarilla mantendrá hidr...
🔎 Analizando post de 25

In [ ]:
import requests
import pandas as pd
import time

# --- TUS DATOS ---
# --- TUS DATOS ---
ACCESS_TOKEN = 'EAAQ9FrwwOUABQiVgUJzxNVfpEfoW22ukZBGn21izTqT7oZBFUcoJu0k584WDIQPS2XuAqZAZByWyE2rb5kpq5ZBc2JreZCNNHV1Q1cC4pFlZBRQ2Ov6nySEiOXtrVQZAIhkAcjVAPh4DoZBmVDnSectPM42AC157AV82Ou9vS2t9hk7U0MWeYrpRrTxcW20QeksjR2Cqb7uImw4jh2CZC9k3oPTPsUNGxlxy9nRm9D2cmsGd9PtupaZBjIwtbEDRxskbvaImjfNzakaMRJ7VPDK8eZCQR5j0'
# Asegúrate de usar el IG User ID (no el de Facebook)
IG_USER_ID = '17841402208128553' 

def obtener_comentarios_videos_ig():
    # 1. Pedimos 'media_type' para distinguir videos de fotos
    # Pedimos limit=100 para tener un buen lote inicial y buscar los videos ahí
    url_media = f"https://graph.facebook.com/v24.0/{IG_USER_ID}/media?fields=id,caption,media_type,timestamp&limit=100&access_token={ACCESS_TOKEN}"
    
    data_final = []
    contador_videos = 0
    LIMITE_VIDEOS = 25 # Queremos exactamente 25 videos
    
    print(f"🚀 Buscando los últimos {LIMITE_VIDEOS} videos...")

    while url_media and contador_videos < LIMITE_VIDEOS:
        try:
            res_media = requests.get(url_media).json()
            items = res_media.get('data', [])
            
            if not items:
                break # No hay más publicaciones

            for item in items:
                # --- FILTRO: Solo procesamos si es VIDEO ---
                if item.get('media_type') == 'VIDEO':
                    contador_videos += 1
                    
                    media_id = item['id']
                    # Usamos los primeros 50 caracteres del caption como "Nombre del Video"
                    nombre_video = item.get('caption', 'Video sin descripción')[:50].replace('\n', ' ')
                    
                    print(f"[{contador_videos}/{LIMITE_VIDEOS}] Extrayendo comentarios de: {nombre_video}...")

                    # 2. Obtener comentarios de este video
                    url_comm = f"https://graph.facebook.com/v24.0/{media_id}/comments?fields=text,username,timestamp&limit=100&access_token={ACCESS_TOKEN}"
                    
                    while url_comm:
                        res_comm = requests.get(url_comm).json()
                        comentarios = res_comm.get('data', [])
                        
                        for c in comentarios:
                            data_final.append({
                                'ID_Video': media_id,
                                'Nombre_Video': item.get('caption'), # El "Nombre" completo
                                'Fecha_Video': item.get('timestamp'),
                                'Usuario_Comentario': c.get('username'), # Persona que comentó
                                'Comentario': c.get('text'),
                                'Fecha_Comentario': c.get('timestamp')
                            })
                        
                        # Paginación de comentarios
                        url_comm = res_comm.get('paging', {}).get('next')

                    # Si ya llegamos a 25 videos, rompemos el bucle for
                    if contador_videos >= LIMITE_VIDEOS:
                        break
            
            # Paginación de publicaciones (Siguiente lote si no hemos llegado a 25 videos)
            url_media = res_media.get('paging', {}).get('next')

        except Exception as e:
            print(f"Error: {e}")
            break

    return data_final

# Ejecución
print("Iniciando proceso...")
comentarios = obtener_comentarios_videos_ig()

if comentarios:
    df = pd.DataFrame(comentarios)
    # Reordenamos columnas para que se vea ordenado en Excel
    columnas = ['ID_Video', 'Nombre_Video', 'Fecha_Video', 'Usuario_Comentario', 'Comentario', 'Fecha_Comentario']
    df = df[columnas]
    
    df.to_excel('comentarios_ultimos_25_videos.xlsx', index=False)
    print(f"✅ ¡Listo! Se guardaron {len(comentarios)} comentarios de los últimos 25 videos.")
else:
    print("No se encontraron comentarios o hubo un error con el token.")

# Hora de actividad

In [ ]:
import requests

PAGE_ID = '895431323874532'
PAGE_TOKEN = 'EAAQ9FrwwOUABQstLGKh6i8gjREfyFEN607oerhsuTYZBVxDWJhKTLIHRCMlvkA229uOMdRftZBkpG4tlwHuyPHnPuMLy6wiZABJVWseW2Kv1NZAOEsxrh2YREEcT9G9NQTjhpI2BSwZAMQKIkC5Fd2XfnpmWdwdm2nHv38rU6YZAHsjD4Phu47vdZBZBLmi1oP9p0l7gngTlZB4V4KAEyY8DwXvv3O1KfOdAchWbrgzQH6lYs' # El que obtuviste de me/accounts

def test_all_metrics():
    # Lista de métricas de menor a mayor restricción
    metrics = [
        ('page_impressions', 'day'),          # Pública/Básica (Debería funcionar)
        ('page_post_engagements', 'day'),     # Interacción (Debería funcionar)
        ('page_views_total', 'day'),          # Visitas (Requiere permisos de análisis)
        ('page_fans_online', 'lifetime')      # Privada/Audiencia (La que está fallando)
    ]
    
    print(f"--- Iniciando Diagnóstico para ID: {PAGE_ID} ---\n")
    
    for metric, period in metrics:
        url = f"https://graph.facebook.com/v21.0/{PAGE_ID}/insights"
        params = {
            'metric': metric,
            'period': period,
            'access_token': PAGE_TOKEN
        }
        
        response = requests.get(url, params=params)
        res_json = response.json()
        
        if 'data' in res_json and len(res_json['data']) > 0:
            print(f"✅ [OK] {metric}: Funciona correctamente.")
        else:
            error_msg = res_json.get('error', {}).get('message', 'Sin datos disponibles')
            print(f"❌ [ERROR] {metric}: {error_msg}")

if __name__ == "__main__":
    test_all_metrics()

In [ ]:
import requests
import pandas as pd

PAGE_ID = '895431323874532'
PAGE_TOKEN =  'EAAQ9FrwwOUABQstLGKh6i8gjREfyFEN607oerhsuTYZBVxDWJhKTLIHRCMlvkA229uOMdRftZBkpG4tlwHuyPHnPuMLy6wiZABJVWseW2Kv1NZAOEsxrh2YREEcT9G9NQTjhpI2BSwZAMQKIkC5Fd2XfnpmWdwdm2nHv38rU6YZAHsjD4Phu47vdZBZBLmi1oP9p0l7gngTlZB4V4KAEyY8DwXvv3O1KfOdAchWbrgzQH6lYs' # El que obtuviste de me/accounts


def extraer_exito_marketing():
    # Usamos la métrica que ya confirmamos que SÍ funciona
    url = f"https://graph.facebook.com/v21.0/{PAGE_ID}/insights"
    params = {
        'metric': 'page_post_engagements',
        'period': 'day',
        'access_token': PAGE_TOKEN
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if 'data' in data:
        # Aquí tienes la data real para mostrar en tu presentación
        print("✅ Datos de Engagement listos para procesar.")
        # Puedes exportar esto a Excel como hicimos antes
    else:
        print("Error al extraer engagement")

    print(data)

extraer_exito_marketing()

In [ ]:
import requests
import pandas as pd
from datetime import datetime

# --- CONFIGURACIÓN ---
PAGE_ID = '895431323874532'
PAGE_TOKEN = 'EAAQ9FrwwOUABQstLGKh6i8gjREfyFEN607oerhsuTYZBVxDWJhKTLIHRCMlvkA229uOMdRftZBkpG4tlwHuyPHnPuMLy6wiZABJVWseW2Kv1NZAOEsxrh2YREEcT9G9NQTjhpI2BSwZAMQKIkC5Fd2XfnpmWdwdm2nHv38rU6YZAHsjD4Phu47vdZBZBLmi1oP9p0l7gngTlZB4V4KAEyY8DwXvv3O1KfOdAchWbrgzQH6lYs'

def extraer_y_exportar_marketing(dias=30):
    url = f"https://graph.facebook.com/v21.0/{PAGE_ID}/insights"
    params = {
        'metric': 'page_post_engagements',
        'period': 'day',
        'since': f'today - {dias} days', 
        'access_token': PAGE_TOKEN
    }
    
    print("🚀 Consultando a Meta Graph API...")
    response = requests.get(url, params=params)
    data = response.json()
    
    if 'data' in data and len(data['data']) > 0:
        # Extraemos los valores de la métrica
        valores_engagement = data['data'][0]['values']
        
        lista_final = []
        for entrada in valores_engagement:
            # Limpiamos la fecha para que sea legible (YYYY-MM-DD)
            fecha_sucia = entrada['end_time']
            fecha_limpia = fecha_sucia.split('T')[0]
            
            lista_final.append({
                'Fecha': fecha_limpia,
                'Interacciones (Engagement)': entrada['value']
            })
        
        # Crear el DataFrame
        df = pd.DataFrame(lista_final)
        
        # --- CÁLCULOS PARA EL EQUIPO DE MARKETING ---
        promedio = df['Interacciones (Engagement)'].mean()
        maximo = df['Interacciones (Engagement)'].max()
        
        print(f"✅ Procesamiento completado.")
        print(f"📊 Promedio de interacción diaria: {promedio:.2f}")
        print(f"🏆 Pico máximo de interacción: {maximo}")
        
        # Exportar a Excel
        nombre_archivo = f"Reporte_Engagement_{PAGE_ID}.xlsx"
        df.to_excel(nombre_archivo, index=False)
        print(f"📁 Archivo guardado como: {nombre_archivo}")
        
        return df
    else:
        print("❌ No se encontraron datos o el token expiró.")
        if 'error' in data:
            print(f"Detalle del error: {data['error']['message']}")
        return None

if __name__ == "__main__":
    df_resultados = extraer_y_exportar_marketing()
    if df_resultados is not None:
        print("\n--- VISTA PREVIA DE LOS DATOS ---")
        print(df_resultados.head())

In [ ]:
import requests
import pandas as pd
from datetime import datetime
import pytz

# --- CONFIGURACIÓN ---
PAGE_ID = '895431323874532'
PAGE_TOKEN = 'EAAQ9FrwwOUABQstLGKh6i8gjREfyFEN607oerhsuTYZBVxDWJhKTLIHRCMlvkA229uOMdRftZBkpG4tlwHuyPHnPuMLy6wiZABJVWseW2Kv1NZAOEsxrh2YREEcT9G9NQTjhpI2BSwZAMQKIkC5Fd2XfnpmWdwdm2nHv38rU6YZAHsjD4Phu47vdZBZBLmi1oP9p0l7gngTlZB4V4KAEyY8DwXvv3O1KfOdAchWbrgzQH6lYs'
ZONA_HORARIA = pytz.timezone('America/Bogota') 

def analizar_interacciones_por_hora(limite_posts=60):
    url = f"https://graph.facebook.com/v21.0/{PAGE_ID}/posts"
    # Añadimos explícitamente el campo created_time dentro de las reacciones
    params = {
        'fields': 'comments.limit(100){created_time},reactions.limit(100){created_time}',
        'limit': limite_posts,
        'access_token': PAGE_TOKEN
    }
    
    print(f"🚀 Analizando interacciones de los últimos {limite_posts} posts...")
    response = requests.get(url, params=params)
    data = response.json()
    posts = data.get('data', [])
    
    registro_tiempos = []

    for post in posts:
        # Extraer horas de comentarios (suelen ser los más consistentes)
        comments = post.get('comments', {}).get('data', [])
        for comm in comments:
            time = comm.get('created_time')
            if time:
                registro_tiempos.append(time)
            
        # Extraer horas de reacciones con manejo de error
        reactions = post.get('reactions', {}).get('data', [])
        for react in reactions:
            # Usamos .get() para evitar el KeyError si el campo no existe
            time = react.get('created_time')
            if time:
                registro_tiempos.append(time)

    if not registro_tiempos:
        print("❌ No se encontraron timestamps en las interacciones.")
        return

    # Procesamiento de datos
    df = pd.DataFrame(registro_tiempos, columns=['timestamp_utc'])
    df['dt_utc'] = pd.to_datetime(df['timestamp_utc'])
    df['dt_local'] = df['dt_utc'].dt.tz_convert(ZONA_HORARIA)
    df['Hora_del_Dia'] = df['dt_local'].dt.hour
    
    # Agrupación por hora
    reporte_horas = df['Hora_del_Dia'].value_counts().sort_index().reset_index()
    reporte_horas.columns = ['Hora', 'Cantidad_Interacciones']

    print("\n📊 ANÁLISIS POR HORA COMPLETADO:")
    print(reporte_horas)
    
    reporte_horas.to_excel("Analisis_Horas_Interaccion_Final.xlsx", index=False)
    return reporte_horas

if __name__ == "__main__":
    reporte = analizar_interacciones_por_hora()

In [ ]:
import requests
import pandas as pd

# CONFIGURACIÓN
PAGE_ID = '895431323874532'
PAGE_TOKEN =  'EAAQ9FrwwOUABQstLGKh6i8gjREfyFEN607oerhsuTYZBVxDWJhKTLIHRCMlvkA229uOMdRftZBkpG4tlwHuyPHnPuMLy6wiZABJVWseW2Kv1NZAOEsxrh2YREEcT9G9NQTjhpI2BSwZAMQKIkC5Fd2XfnpmWdwdm2nHv38rU6YZAHsjD4Phu47vdZBZBLmi1oP9p0l7gngTlZB4V4KAEyY8DwXvv3O1KfOdAchWbrgzQH6lYs' # El que obtuviste de me/accounts


def generar_reporte_vistas():
    url = f"https://graph.facebook.com/v21.0/{PAGE_ID}/insights"
    params = {
        'metric': 'page_views_total',
        'period': 'day',
        'access_token': PAGE_TOKEN
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    if 'data' in data and len(data['data']) > 0:
        vistas = []
        for entry in data['data'][0]['values']:
            vistas.append({
                'Fecha': entry['end_time'][:10],
                'Cantidad_Vistas': entry['value']
            })
        
        df = pd.DataFrame(vistas)
        print("--- REPORTE DE VISTAS DE PÁGINA (Últimos días) ---")
        print(df)
        
        # Guardar para la presentación
        df.to_excel("Reporte_Vistas_LaPocion.xlsx", index=False)
        print("\n✅ Archivo Excel generado para la reunión.")
    else:
        print("No se pudieron extraer datos de vistas.")

if __name__ == "__main__":
    generar_reporte_vistas()